# Subway Congestion Reconstruction

승하차 인원과 공식 혼잡도 비율을 활용하여 시간대별 기준 수용 인원을 추정하고,  
inner / outer 혼잡도를 재구성한다.

In [ ]:
import os
import csv
import zipfile
import chardet
import numpy as np
import pandas as pd

# 환경 설정
BASE_DIR = "."
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")  # 원본 파일 위치
OUT_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(OUT_DIR, exist_ok=True)

# 입력 폴더(역별 파일 있는 곳)
BOARD_DIR = os.path.join(RAW_DIR, "boardings")
CONG_DIR  = os.path.join(RAW_DIR, "congestions")

# 출력 폴더 (역별)
STATION_OUT_DIR = os.path.join(OUT_DIR, "stations")
os.makedirs(STATION_OUT_DIR, exist_ok=True)

# 호선별 열차 수용 용량
LINE_CAPACITY = {1:1600, 2:1600, 3:1600, 4:1600, 5:1280, 6:1280, 7:1280, 8:960}

def get_capacity(line: int) -> int:
    return LINE_CAPACITY.get(int(line), 1600)

In [ ]:
# 파일 인코딩/구분자 자동 감지
def detect_encoding(path: str, sample_size: int = 10000) -> str:
    raw = open(path, "rb").read(sample_size)
    return chardet.detect(raw)["encoding"] or "utf-8"

def detect_delimiter(path: str, encoding: str, sample_size: int = 2048) -> str:
    txt = open(path, encoding=encoding, errors="ignore").read(sample_size)
    try:
        return csv.Sniffer().sniff(txt, delimiters=[",", "\t", ";", "|"]).delimiter
    except Exception:
        return ","

def read_csv_auto(path: str) -> pd.DataFrame:
    enc = detect_encoding(path)
    sep = detect_delimiter(path, enc)
    return pd.read_csv(path, encoding=enc, sep=sep, engine="python")

In [ ]:
# 역 단위 처리 함수
def process_station(board_path: str, cong_path: str, output_path: str) -> None:
    """
    입력:
      - board_path: {역명}_승하차.csv
      - cong_path : {역명}_혼잡도.csv
    출력:
      - {역명}_내외선_승하차_혼잡도.csv
    """
    # 파일 로드
    bd = read_csv_auto(board_path)
    cd = read_csv_auto(cong_path)

    # 컬럼 정리 및 이름 통일
    bd.columns = bd.columns.str.strip()
    cd.columns = cd.columns.str.strip()

    bd = bd.rename(columns={
        "Datetime": "날짜",
        "Line": "호선",
        "구분": "승하차 구분",
        "역명": "역명"
    })

    # 호선 처리 (숫자만 추출)
    if "호선" not in bd.columns:
        raise ValueError("승하차 파일에 호선(Line) 정보가 없습니다.")
    bd["호선"] = bd["호선"].astype(str).str.extract(r"(\d+)").astype(int)

    # 혼잡도 파일 쪽 호선 컬럼 통일
    if "호선" not in cd.columns and "Line" in cd.columns:
        cd["호선"] = cd["Line"]
    cd["호선"] = cd["호선"].astype(str).str.extract(r"(\d+)").astype(int)

    # 상하구분 통일 
    if "상하구분" in cd.columns:
        cd["상하구분"] = cd["상하구분"].replace({"상행": "상선", "하행": "하선"})
    else:
        raise ValueError("혼잡도 파일에 상하구분 컬럼이 없습니다.")

    # 승하차 구분 검사
    bd["승하차 구분"] = bd["승하차 구분"].astype(str).str.strip()
    if not set(["승차", "하차"]).intersection(set(bd["승하차 구분"].unique())):
        raise ValueError(f"{board_path}에 '승차' 또는 '하차' 구분이 없습니다.")

    # 승하차 데이터를 long 형태로 변환 후 시간 추출
    time_cols_b = [c for c in bd.columns if ":" in c]  # "06:00~06:30" 같은 컬럼을 가정
    if not time_cols_b:
        raise ValueError("승하차 파일에서 시간대 컬럼(: 포함)을 찾지 못했습니다.")

    bl = bd.melt(
        id_vars=["날짜", "호선", "승하차 구분"],
        value_vars=time_cols_b,
        var_name="time_bin",
        value_name="count",
    ).dropna(subset=["count"])

    bl["hour"] = bl["time_bin"].str.extract(r"(\d+)").astype(int)
    bsum = bl.groupby(["날짜", "호선", "hour", "승하차 구분"], as_index=False)["count"].sum()

    bp = (
        bsum.pivot(index=["날짜", "호선", "hour"], columns="승하차 구분", values="count")
        .fillna(0)
        .reset_index()
    )
    for col in ["승차", "하차"]:
        if col not in bp.columns:
            bp[col] = 0

    # 혼잡도 비율 데이터 정리 (요일구분/상하구분 + 시간 컬럼을 long으로 변환)
    if "요일구분" not in cd.columns:
        raise ValueError("혼잡도 파일에 요일구분 컬럼이 없습니다.")

    time_cols_c = [c for c in cd.columns if ":" in c]
    if not time_cols_c:
        raise ValueError("혼잡도 파일에서 시간대 컬럼(: 포함)을 찾지 못했습니다.")

    cl = cd.melt(
        id_vars=["호선", "요일구분", "상하구분"],
        value_vars=time_cols_c,
        var_name="time_str",
        value_name="cong",
    ).dropna(subset=["cong"])

    cl["hour"] = cl["time_str"].str.extract(r"(\d+)").astype(int)
    cs = cl.groupby(["호선", "요일구분", "hour", "상하구분"], as_index=False)["cong"].sum()

    cp = (
        cs.pivot(index=["호선", "요일구분", "hour"], columns="상하구분", values="cong")
        .fillna(0)
        .reset_index()
    )
    cp.columns.name = None
    for col in ["상선", "하선"]:
        if col not in cp.columns:
            cp[col] = 0

    # inner/outer 비율 계산 (상선:하선 비율)
    cp["inner_ratio"] = np.where((cp["상선"] + cp["하선"]) > 0, cp["상선"] / (cp["상선"] + cp["하선"]), 0.5)
    cp["outer_ratio"] = 1 - cp["inner_ratio"]

    # 날짜 → 요일구분으로 매핑 
    bp["date"] = pd.to_datetime(bp["날짜"], errors="coerce")
    bp["weekday"] = bp["date"].dt.weekday

    def map_weekday(w):
        if w < 5:
            return "평일"
        if w == 5:
            return "토요일"
        return "일요일/공휴일"

    bp["요일구분"] = bp["weekday"].apply(map_weekday)

    # merge 후 inner/outer 승하차 분리 및 혼잡도 계산
    df = pd.merge(bp, cp, on=["호선", "요일구분", "hour"], how="left")
    df[["inner_ratio", "outer_ratio"]] = df[["inner_ratio", "outer_ratio"]].fillna(0.5)

    df["inner_ride"]   = (df["승차"] * df["inner_ratio"]).round().astype(int)
    df["inner_getoff"] = (df["하차"] * df["inner_ratio"]).round().astype(int)
    df["outer_ride"]   = (df["승차"] * df["outer_ratio"]).round().astype(int)
    df["outer_getoff"] = (df["하차"] * df["outer_ratio"]).round().astype(int)

    df["capacity"] = df["호선"].apply(get_capacity)
    df["inner_congestion_pct"] = (df["inner_ride"] / df["capacity"] * 100).round(2)
    df["outer_congestion_pct"] = (df["outer_ride"] / df["capacity"] * 100).round(2)

    out_cols = [
        "날짜", "호선", "hour",
        "inner_ride", "inner_getoff",
        "outer_ride", "outer_getoff",
        "inner_congestion_pct", "outer_congestion_pct"
    ]
    df[out_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {output_path}")

In [ ]:
# 일괄 처리(역 폴더 기준 자동 탐색)
def list_station_files(board_dir: str, cong_dir: str):
    """boardings/와 congestions/에 모두 존재하는 역 파일 목록을 만든다."""
    if not os.path.isdir(board_dir) or not os.path.isdir(cong_dir):
        raise FileNotFoundError("입력 폴더(boardings/congestions)가 없습니다.")

    boards = {f.replace("_승하차.csv", ""): os.path.join(board_dir, f)
              for f in os.listdir(board_dir) if f.endswith("_승하차.csv")}
    congs  = {f.replace("_혼잡도.csv", ""): os.path.join(cong_dir, f)
              for f in os.listdir(cong_dir) if f.endswith("_혼잡도.csv")}

    common = sorted(set(boards.keys()) & set(congs.keys()))
    return common, boards, congs

stations, board_map, cong_map = list_station_files(BOARD_DIR, CONG_DIR)
print(f"처리 대상 역 수: {len(stations)}")

for st in stations:
    out_path = os.path.join(STATION_OUT_DIR, f"{st}_내외선_승하차_혼잡도.csv")
    try:
        process_station(board_map[st], cong_map[st], out_path)
    except Exception as e:
        print(f"실패({st}): {e}")

In [ ]:
# 결과를 zip으로 묶기
ZIP_OUT = os.path.join(OUT_DIR, "역별_내외선_승하차_혼잡도.zip")
with zipfile.ZipFile(ZIP_OUT, "w") as zf:
    for f in os.listdir(STATION_OUT_DIR):
        if f.endswith(".csv"):
            zf.write(os.path.join(STATION_OUT_DIR, f), arcname=f)

print("zip 생성:", ZIP_OUT)